# Lab II - mdm26 + cambios de la versión 22

**Integrantes:** COMPLETAR

Base: notebook `mdm26` del profesor. Se agregan los cambios 01, 02, 04, 12, 13, 14 y 22 (ver `docs/CELDAS_MEJOR_VERSION.md`),
la validación por rescatista y los dos preprocesados del profesor. Las celdas tocadas tienen `# CAMBIO`.

**Inputs que hay que agregar en Kaggle:**
1. competencia `petfinder-adoption-prediction`;
2. notebook `nickel/examen-lab-ii-preprocesado-texto`;
3. notebook `nickel/examen-lab-ii-preprocesado-img`.

Las rutas se buscan solas (celda `CAMBIO RUTAS`).

In [ ]:
import pandas as pd
import numpy as np                                          # CAMBIO 01

from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedGroupKFold   # CAMBIO VALIDACIÓN: folds por rescatista
from sklearn.metrics import cohen_kappa_score

from lightgbm import LGBMClassifier, LGBMRegressor, early_stopping   # CAMBIO 01: LGBMRegressor
from scipy.optimize import minimize                         # CAMBIO 01: para optimizar los umbrales

from tqdm.notebook import tqdm

In [ ]:
def metric(y_true, y_pred):
    res = cohen_kappa_score(y_true, y_pred.reshape((y_true.shape[0], 5), order="F").argmax(axis=1), weights= 'quadratic')
    return "kappa", res, True

In [ ]:
# CAMBIO RUTAS: busca los inputs dondequiera que Kaggle los haya montado
# (Kaggle ahora usa /kaggle/input/competitions/... ; así no depende del nombre de la carpeta)
import glob, os
RAIZ = "/kaggle/input" if os.path.exists("/kaggle/input") else "../input"

encontrados = glob.glob(f"{RAIZ}/**/train/train.csv", recursive=True)
assert encontrados, "No encuentro train.csv: agregá la competencia petfinder-adoption-prediction como input"
COMP = os.path.dirname(os.path.dirname(encontrados[0]))

TXT = IMG = None
for ruta in glob.glob(f"{RAIZ}/**/train.parquet", recursive=True):   # se reconocen por sus columnas
    columnas = pd.read_parquet(ruta).columns
    if "img_feat_1" in columnas:
        IMG = os.path.dirname(ruta)
    elif "joy" in columnas:
        TXT = os.path.dirname(ruta)
assert IMG, "Falta el input 'examen Lab II - preprocesado img' (nickel/examen-lab-ii-preprocesado-img)"
assert TXT, "Falta el input 'examen Lab II - preprocesado texto' (nickel/examen-lab-ii-preprocesado-texto)"
print("Competencia:", COMP)
print("Texto:      ", TXT)
print("Imágenes:   ", IMG)

In [ ]:
train = pd.read_csv(f"{COMP}/train/train.csv", index_col="PetID")     # CAMBIO RUTAS: ruta encontrada arriba
test = pd.read_csv(f"{COMP}/test/test.csv", index_col="PetID")
if os.path.exists(f"{COMP}/test/sample_submission.csv"):               # CAMBIO RUTAS: no se usa; en la copia local no está
    ss = pd.read_csv(f"{COMP}/test/sample_submission.csv", index_col="PetID")
# train = pd.read_csv("/kaggle/input/competitions/petfinder-adoption-prediction/train/train.csv", index_col="PetID")   # original

# CAMBIO VALIDACIÓN: folds agrupados por rescatista (el mismo RescuerID nunca está en train y valid)
# y estratificados por AdoptionSpeed, semilla 42.
folds = [
    train.index[test_idx]
    for train_idx, test_idx in StratifiedGroupKFold(5, shuffle=True, random_state=42).split(
        train, train["AdoptionSpeed"], groups=train["RescuerID"])
]
# folds = [train.index[test_idx] for train_idx, test_idx in KFold(5).split(train)]   # original

In [ ]:
label_col = "AdoptionSpeed"

In [ ]:
train["RescuerID"] = train["RescuerID"].astype("category")
test["RescuerID"] = test["RescuerID"].astype("category")

In [ ]:
train["Name"] = train["Name"].astype("category")
test["Name"] = test["Name"].astype("category")

In [ ]:
# CAMBIO ARREGLO: el original daba error (fillna con una categoría nueva en una columna categórica).
# Se agrega la categoría antes de rellenar; solo es exploración, no cambia el modelo.
train["Name"].cat.add_categories("NU:LLLLLL").fillna("NU:LLLLLL").value_counts().head(10)
# train["Name"].fillna("NU:LLLLLL").value_counts().head(10)   # original

In [ ]:
v = pd.crosstab(train["Name"].str.lower().fillna("no name").str.contains("no name"), train[label_col])
v / v.sum(axis=1).values[:, None]

In [ ]:
v = train[["Name", label_col]].join(train["Name"].value_counts().rename('name_count'), on="Name")
v = pd.crosstab(pd.qcut(v["name_count"], 10, duplicates="drop"), v[label_col])
v / v.sum(axis=1).values[:, None]

In [ ]:
train["Name"]

In [ ]:
# CAMBIO 02: cantidad de mascotas publicadas por cada rescatista.
# Cada conjunto se cuenta por separado (ningún rescatista está en train y test a la vez).
train["RescuerID_count"] = train.groupby("RescuerID", observed=True)["RescuerID"].transform("size")
test["RescuerID_count"] = test.groupby("RescuerID", observed=True)["RescuerID"].transform("size")


In [ ]:
# CAMBIO 04: puntaje medio de las etiquetas de Google Vision en la PRIMERA foto
import glob, json, os

def unir(df, nuevas):
    # CAMBIO ARREGLO: quita antes las columnas que ya estén, así la celda se puede correr dos veces sin error
    nuevas = nuevas.to_frame() if isinstance(nuevas, pd.Series) else nuevas
    return df.drop(columns=nuevas.columns, errors="ignore").join(nuevas)

def score_primera_foto(carpeta):
    valores = {}
    for ruta in glob.glob(os.path.join(carpeta, "*-1.json")):      # solo "<PetID>-1.json"
        pet_id = os.path.basename(ruta)[:-len("-1.json")]
        with open(ruta, encoding="utf-8") as f:
            etiquetas = [e.get("score", 0) for e in json.load(f).get("labelAnnotations", [])]
        valores[pet_id] = np.mean(etiquetas) if etiquetas else np.nan
    return pd.Series(valores, name="meta1_label_score_mean")

train = unir(train, score_primera_foto(f"{COMP}/train_metadata"))
test = unir(test, score_primera_foto(f"{COMP}/test_metadata"))


In [ ]:
# CAMBIO 13: brillo, nitidez, aspecto y saturación de las fotos con OpenCV, agregados por mascota (mean, max, primera foto)
import cv2
from joblib import Parallel, delayed

def stats_foto(ruta):
    pet_id, n = os.path.basename(ruta)[:-len(".jpg")].rsplit("-", 1)     # archivo "<PetID>-<n>.jpg"
    img = cv2.imread(ruta)
    if img is None:
        return {"PetID": pet_id, "foto": int(n)}
    gris = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    alto, ancho = gris.shape
    return {"PetID": pet_id, "foto": int(n),
            "brillo": float(gris.mean()),
            "nitidez": float(cv2.Laplacian(gris, cv2.CV_64F).var()),
            "aspecto": ancho / alto,
            "saturacion": float(hsv[:, :, 1].mean())}

VARIABLES_CV = ["brillo", "nitidez", "aspecto", "saturacion"]

def opencv_por_mascota(carpeta):
    rutas = glob.glob(os.path.join(carpeta, "*.jpg"))
    fotos = pd.DataFrame(Parallel(n_jobs=-1, batch_size=256)(delayed(stats_foto)(r) for r in rutas))
    g = fotos.groupby("PetID")[VARIABLES_CV]
    media = g.mean().add_prefix("img_cv_mean_")
    maximo = g.max().add_prefix("img_cv_max_")
    primera = fotos[fotos["foto"] == 1].set_index("PetID")[VARIABLES_CV].add_prefix("img_cv_first_")
    return media.join(maximo).join(primera)

train = unir(train, opencv_por_mascota(f"{COMP}/train_images"))
test = unir(test, opencv_por_mascota(f"{COMP}/test_images"))
print("Con fotos: train", train["img_cv_mean_brillo"].notna().mean().round(3),
      "| test", test["img_cv_mean_brillo"].notna().mean().round(3))


In [ ]:
# CAMBIO 14: agregados por mascota de Google Vision (dominantColors, labelAnnotations, cropHints), todas las fotos
def valores_foto(ruta):
    # Google omite los campos que valen 0, por eso el valor por defecto es 0.
    pet_id = os.path.basename(ruta)[:-len(".json")].rsplit("-", 1)[0]
    with open(ruta, encoding="utf-8") as f:
        m = json.load(f)
    colores = m.get("imagePropertiesAnnotation", {}).get("dominantColors", {}).get("colors", [])
    recortes = m.get("cropHintsAnnotation", {}).get("cropHints", [])
    return (pet_id,
            [c.get("score", 0) for c in colores],
            [c.get("pixelFraction", 0) for c in colores],
            [e.get("score", 0) for e in m.get("labelAnnotations", [])],
            [r.get("confidence", 0) for r in recortes],
            [r.get("importanceFraction", 0) for r in recortes])

def vision_por_mascota(carpeta):
    fotos = Parallel(n_jobs=-1, batch_size=512)(delayed(valores_foto)(r) for r in glob.glob(os.path.join(carpeta, "*.json")))
    nombres = ["dom_score", "dom_pixfrac", "label_score", "crop_conf", "crop_importance"]
    filas = []
    for pet_id, *listas in fotos:                     # una fila por valor individual
        for nombre, lista in zip(nombres, listas):
            filas += [(pet_id, nombre, v) for v in lista]
    largo = pd.DataFrame(filas, columns=["PetID", "variable", "valor"])
    agg = largo.groupby(["PetID", "variable"])["valor"].agg(["mean", "max"]).unstack("variable")
    agg.columns = [f"vis_{var}_{estad}" for estad, var in agg.columns]
    pedidas = ["vis_dom_score_mean", "vis_dom_score_max", "vis_dom_pixfrac_mean", "vis_dom_pixfrac_max",
               "vis_label_score_max",
               "vis_crop_conf_mean", "vis_crop_conf_max", "vis_crop_importance_mean", "vis_crop_importance_max"]
    return agg[pedidas]

train = unir(train, vision_por_mascota(f"{COMP}/train_metadata"))
test = unir(test, vision_por_mascota(f"{COMP}/test_metadata"))


In [ ]:
# CAMBIO 22: Word2Vec sobre Description, entrenado SOLO con el train de cada fold (se usa en el entrenamiento)
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess
import zlib

# Tokens guardados aparte: el select_dtypes(exclude="O") de la celda siguiente saca la columna de texto
tokens_train = train["Description"].fillna("").astype(str).map(simple_preprocess)
tokens_test = test["Description"].fillna("").astype(str).map(simple_preprocess)

W2V_DIM = 100
W2V_COLS = [f"w2v_{k}" for k in range(W2V_DIM)]

def hash_fijo(texto):   # el hash de Python cambia en cada corrida; crc32 no -> resultado reproducible
    return zlib.crc32(texto.encode("utf-8"))

def vector_promedio(tokens, modelo):
    conocidas = [t for t in tokens if t in modelo.wv.key_to_index]
    return modelo.wv[conocidas].mean(axis=0) if conocidas else np.zeros(W2V_DIM)

def agregar_w2v(X_tr, X_va, X_te, tok_tr, tok_va, tok_te):
    modelo = Word2Vec(sentences=list(tok_tr), vector_size=W2V_DIM, window=5, min_count=2, workers=1, seed=42,
                      hashfxn=hash_fijo)
    salida = []
    for X, tok in ((X_tr, tok_tr), (X_va, tok_va), (X_te, tok_te)):
        vectores = pd.DataFrame(np.vstack([vector_promedio(t, modelo) for t in tok]), index=X.index, columns=W2V_COLS)
        salida.append(pd.concat([X, vectores], axis=1))
    return salida


In [ ]:
# CAMBIO PREPROCESADOS: emociones del texto y embeddings de la primera foto (notebooks del profesor), unidos por PetID
train = unir(unir(train, pd.read_parquet(f"{TXT}/train.parquet")), pd.read_parquet(f"{IMG}/train.parquet"))
test = unir(unir(test, pd.read_parquet(f"{TXT}/test.parquet")), pd.read_parquet(f"{IMG}/test.parquet"))
print("Con embedding de imagen: train", train["img_feat_1"].notna().mean().round(3), "| test", test["img_feat_1"].notna().mean().round(3))

In [ ]:
# CAMBIO 12: semillas a promediar en cada fold
SEMILLAS = [0, 1, 2, 3, 4]

fold_preds = []
test_preds = []
for i, fold in enumerate(folds):
    X_train = train.drop(fold).drop("AdoptionSpeed", axis=1).select_dtypes(exclude="O")
    y_train = train.drop(fold)["AdoptionSpeed"]

    X_valid = train.loc[fold].drop("AdoptionSpeed", axis=1).select_dtypes(exclude="O")
    y_valid = train.loc[fold, "AdoptionSpeed"]

    # CAMBIO 22: Word2Vec entrenado con las descripciones del train de este fold; vectores para train, valid y test
    X_train, X_valid, X_test = agregar_w2v(
        X_train, X_valid, test[X_train.columns],
        tokens_train.loc[X_train.index], tokens_train.loc[X_valid.index], tokens_test.loc[test.index])

    # CAMBIO 12: un modelo por semilla, con 80% de filas y columnas al azar; se promedian las predicciones
    pred_valid, pred_test = [], []
    for semilla in SEMILLAS:
        # CAMBIO 01: regresión en vez de clasificación
        learner = LGBMRegressor(random_state=semilla, subsample=0.8, subsample_freq=1, colsample_bytree=0.8)
        # learner = LGBMClassifier()   # original
        # learner.fit(X_train, y_train, )   # original: quitado, el fit siguiente lo reemplazaba

        # CAMBIO 01: sin eval_metric=metric (es para 5 probabilidades); el early stopping usa el error cuadrático
        learner.fit(X_train, y_train, eval_set=[(X_train, y_train), (X_valid, y_valid)],
                    callbacks=[early_stopping(5, verbose=False)])
        pred_valid.append(learner.predict(X_valid))
        pred_test.append(learner.predict(X_test[X_train.columns]))
    fold_preds.append(pd.Series(np.mean(pred_valid, axis=0), index=fold, name="AdoptionSpeed"))
    test_preds.append(pd.Series(np.mean(pred_test, axis=0), index=test.index))

fold_preds = pd.concat(fold_preds)
# CAMBIO 01: promedio de los 5 folds; los umbrales se aplican en la celda siguiente
test_preds = sum(test_preds) / len(test_preds)


In [ ]:
# CAMBIO 01: umbrales optimizados para pasar del número continuo a la clase 0-4
def aplicar_cortes(pred, cortes):
    return np.digitize(pred, np.sort(cortes))     # 4 cortes ordenados -> clases 0..4

def optimizar_cortes(pred, y, inicio=(0.5, 1.5, 2.5, 3.5)):
    perdida = lambda c: -cohen_kappa_score(y, aplicar_cortes(pred, c), weights="quadratic")
    return np.sort(minimize(perdida, inicio, method="nelder-mead").x)

y_oof = train.loc[fold_preds.index, "AdoptionSpeed"]

# Medición honesta: los cortes de cada fold se eligen con los OTROS 4 folds
oof_clases = []
for fold in folds:
    resto = fold_preds.index.difference(fold)
    c = optimizar_cortes(fold_preds[resto], y_oof[resto])
    oof_clases.append(pd.Series(aplicar_cortes(fold_preds[fold], c), index=fold))
oof_clases = pd.concat(oof_clases)
kappas = [cohen_kappa_score(y_oof[f], oof_clases[f], weights="quadratic") for f in folds]
print("Kappa por fold:", [round(k, 4) for k in kappas])
print(f"Kappa media: {np.mean(kappas):.4f} ± {np.std(kappas, ddof=1):.4f}")

# Para la entrega: cortes elegidos con TODO el OOF
cortes = optimizar_cortes(fold_preds, y_oof)
print("Cortes:", np.round(cortes, 3))
test_preds = pd.Series(aplicar_cortes(test_preds, cortes), index=test_preds.index, name="AdoptionSpeed")
test_preds.value_counts().sort_index()


In [ ]:
# CAMBIO 01: el kappa se mide sobre las clases (con los cortes elegidos fuera de cada fold), con pesos cuadráticos
cohen_kappa_score(train.loc[oof_clases.index, "AdoptionSpeed"], oof_clases, weights="quadratic")
# cohen_kappa_score(train.loc[fold_preds.index, "AdoptionSpeed"], fold_preds)   # original

In [ ]:
test_preds.to_csv("submission.csv")
!head submission.csv